In [1]:
from datasets import load_dataset
import pandas as pd

from tqdm import tqdm
from gpt_arcitecture import GPTModel, load_params, download_and_load_gpt2, generate
import torch
import tiktoken
from torch.utils.data import DataLoader, random_split
from torch.nn.functional import cross_entropy

import warnings
warnings.filterwarnings("ignore")

# torch.set_default_device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = generator = torch.Generator(device = device)

tokenizer = tiktoken.get_encoding("gpt2")

settings, vocab, params = download_and_load_gpt2("124M", "./archive")
model = GPTModel().to(device)
load_params(model,params)

eos_id = 50246
ignore_id = -100

2025-11-10 08:34:02.145990: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-10 08:34:02.241827: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-10 08:34:04.595247: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-10 08:34:04.596101: I external/local_xla/xla/tsl/cuda/cudart

File already exists and is up-to-date: ./archive/124M/checkpoint
File already exists and is up-to-date: ./archive/124M/encoder.json
File already exists and is up-to-date: ./archive/124M/hparams.json
File already exists and is up-to-date: ./archive/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ./archive/124M/model.ckpt.index
File already exists and is up-to-date: ./archive/124M/model.ckpt.meta
File already exists and is up-to-date: ./archive/124M/vocab.bpe


In [ ]:
train_df = load_dataset("databricks/databricks-dolly-15k")
train_df = train_df["train"].to_pandas()
print(train_df.columns)
print(train_df["category"].value_counts())

train_df = train_df.sample(frac=1).reset_index()

# prefix = "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n"
prefix = ""
train_df["formatted"] = prefix + "### Instruction:\n" + train_df["instruction"] + "\n### Context:\n" + train_df["context"]  + "\n### Response:\n" + train_df["response"]
train_encoded = list(map(tokenizer.encode, train_df["formatted"]))

Index(['instruction', 'context', 'response', 'category'], dtype='object')
category
open_qa                   3742
general_qa                2191
classification            2136
closed_qa                 1773
brainstorming             1766
information_extraction    1506
summarization             1188
creative_writing           709
Name: count, dtype: int64


In [3]:
def custom_collate(batch, allowed_max_len=settings["n_ctx"]):
    B = len(batch)

    lens = [min(len(x) + 1, allowed_max_len + 1) for x in batch]
    L = max(lens)

    # Preallocate once. Fill with sensible defaults.
    inputs  = torch.full((B, L - 1), eos_id,     dtype=torch.long)   # pad inputs with EOS
    targets = torch.full((B, L - 1), ignore_id,  dtype=torch.long)   # pad targets with IGNORE

    for i, seq in enumerate(batch):
        s = seq + [eos_id]
        s = s[:L]

        n = lens[i] - 1                # number of valid positions in inputs/targets
        if n <= 0:
            continue

        inputs[i, :n]  = torch.as_tensor(s[:-1][:n], dtype=torch.long)
        targets[i, :n] = torch.as_tensor(s[1:][:n],  dtype=torch.long)

    return inputs, targets

train_split, test_split, val_split = random_split(train_encoded, lengths = [0.85, 0.10,0.05], generator=generator)
BATCH_SIZE = 1
train_loader = DataLoader(train_split, batch_size = BATCH_SIZE, shuffle = True, generator=generator, drop_last=True , collate_fn=custom_collate)
test_loader = DataLoader(test_split, batch_size = BATCH_SIZE, shuffle = False, generator=generator , collate_fn=custom_collate)
val_loader = DataLoader(val_split, batch_size = BATCH_SIZE, shuffle = False, generator=generator , collate_fn=custom_collate)

In [ ]:
epochs = 10
history = []

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)

for epoch in range(epochs):
    iterations = 0
    total_loss = 0

    for (input_batch, target_batch) in train_loader:
        iterations+=1
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)

        optimizer.zero_grad()

        pred_batch = model(input_batch).to(device)
        loss = cross_entropy(pred_batch.flatten(0,1), target_batch.flatten())
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

        if (iterations)%200 == 0:
            with torch.no_grad():
                avg_train_loss = total_loss/((iterations)*BATCH_SIZE)

                total_val_loss = 0
                for input_batch, target_batch in val_loader:
                    pred_batch = model(input_batch)
                    loss = cross_entropy(pred_batch.flatten(0,1), target_batch.flatten())
                    total_val_loss += loss.item()

                avg_val_loss = total_val_loss / (len(val_loader)*BATCH_SIZE)
                this_epoch_history = {
                    "Epochs": epoch+1,
                    "Samples Seen" : iterations*BATCH_SIZE,
                    "Training Loss": avg_train_loss,
                    "Validation Loss": avg_val_loss,
                }

                history.append(this_epoch_history)

            if (iterations)%1000 == 0:
                print(this_epoch_history)

torch.save(model.state_dict(), "./archive/finetuned.pth")
history_df = pd.DataFrame(history)
history_df.to_csv("./archive/finetuned.csv", index=False)

{'Epochs': 1, 'Samples Seen': 200, 'Training Loss': 2.9086916559934615, 'Validation Loss': 2.770562465429306}
{'Epochs': 1, 'Samples Seen': 400, 'Training Loss': 2.8094902412593363, 'Validation Loss': 2.765004426797231}
{'Epochs': 1, 'Samples Seen': 600, 'Training Loss': 2.761490313410759, 'Validation Loss': 2.7746853571732837}
{'Epochs': 1, 'Samples Seen': 800, 'Training Loss': 2.73687842130661, 'Validation Loss': 2.739791256825129}
{'Epochs': 1, 'Samples Seen': 1000, 'Training Loss': 2.728965567886829, 'Validation Loss': 2.710265984296799}
{'Epochs': 1, 'Samples Seen': 1200, 'Training Loss': 2.7263199345270794, 'Validation Loss': 2.698514289855957}
{'Epochs': 1, 'Samples Seen': 1400, 'Training Loss': 2.723845205221857, 'Validation Loss': 2.682855711857478}
{'Epochs': 1, 'Samples Seen': 1600, 'Training Loss': 2.7237380672618747, 'Validation Loss': 2.688908259789149}
{'Epochs': 1, 'Samples Seen': 1800, 'Training Loss': 2.7223105113042725, 'Validation Loss': 2.6982495280106864}
{'Epochs

In [6]:
import plotly.express as px
history_df = pd.read_csv("./archive/finetuned.csv")


fig = px.line(history_df, x = range(len(history_df)), y=["Training Loss", "Validation Loss"], color="Epochs")
fig.show()

fig = px.bar(history_df, x= "Epochs", y = ["Training Loss"])
fig.show()

In [ ]:
inp = "What is the highest GDP country in Asia"
context = "China and India are amongst the top countries in Asia"
output = generate(model, f"### Instruction:\n{inp} \n### Context:\n{context} \n### Response:\n",max_new_tokens=100)
print(output)


model.load_state_dict(torch.load("./archive/finetuned.pth", map_location=torch.device('cpu')))
output = generate(model, f"### Instruction:\n{inp} \n### Context:\n{context} \n### Response:\n",max_new_tokens=100)
print(output.split("### Response:", 1)[1])

### Instruction:
What is the highest GDP country in Asia 
### Context:
China and India are amongst the top countries in Asia 
### Response:
They are the United States of America

1/ Pratao 
2/Jakarta  
3/Chongqing 
4  illustrate Dramatic


In [9]:
instruction = "What will be my spirit animal"
context = "I really like pandas, but am scared by crocodiles. Kiwi's are the most like me."

output = generate(model, f"### Instruction:\n{instruction} \n### Context:\n{context} \n### Response:\n", top_k = 50 ,max_new_tokens=100,eos_id=eos_id)
print(output.split("### Response:", 1)[1])


I really like my favorite part of my little legend. The little figures are a common saying that pandas will be your favorite part of you.  I've lived in the early days of their family and have always lived in their favorite place.  The reason is that I've never believed that Pandas have more than 100,000 pounds in their yearly store.  pandas have a regular happy moment when you love their thing, and my favorite thing to let them love to cook.  It
